In [1]:
# !python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [ ]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
  'daducci_amico',
  'yangjasp_optimall',
  'anybotics_kindr',
  'scikit-fuzzy_scikit-fuzzy',
  'mhhennig_hs2',
  'bids-apps_rshrf',
  'juliastats_lasso.jl',
  'dcc-lab_pyhardwarelibrary',
  'cmillion_gphoton',
  'aim-uofa_adelaidet'
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

KeyError: 'Key 7363696b69742d66757a7a795f7363696b69742d66757a7a796d6868656e6e69675f687332 not found in /da5_fast/p2cFullV2605.22.tch'

In [ ]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 11%|█▏        | 7/62 [00:07<00:55,  1.02s/it]

Got Errors {'231f31b8ddcbb2e79fd1466af9a612dfcfb459c9': 'Key 231f31b8ddcbb2e79fd1466af9a612dfcfb459c9 not found in /da5_fast/All.sha1c/commit_35.tch'}


100%|██████████| 62/62 [01:03<00:00,  1.02s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00534996cc67dc37a0406ba20f8d16a68fa3ff79,21b7f5309dc04f392df524bd61b459eab3a0817f,[d65239cc8bc5fe7da12ca365aaa552792b0bc0ce],nightwnvol <notte_94@hotmail.it>,1656677387,+0200,nightwnvol <notte_94@hotmail.it>,1656677387,+0200,refactor: rename methods and variables\n,00534996cc67dc37a0406ba20f8d16a68fa3ff79,daducci_amico
1,00c6ac9579f55819e5e800aba08f9cf18b314c0a,174834197983965595f4ce17e9b58a916be47070,[09cfbdda847232fd5b85f1cf00b4c5b8c711086f],Alessandro Daducci <alessandro.daducci@univr.it>,1638532643,+0100,Alessandro Daducci <alessandro.daducci@univr.it>,1638532643,+0100,Install information are stored (and taken from...,00c6ac9579f55819e5e800aba08f9cf18b314c0a,daducci_amico


In [ ]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00534996cc67dc37a0406ba20f8d16a68fa3ff79', 'tree': '21b7f5309dc04f392df524bd61b459eab3a0817f', 'parent': ['d65239cc8bc5fe7da12ca365aaa552792b0bc0ce'], 'author': 'nightwnvol <notte_94@hotmail.it>', 'author_time': 1656677387, 'author_tz': '+0200', 'committer': 'nightwnvol <notte_94@hotmail.it>', 'committer_time': 1656677387, 'committer_tz': '+0200', 'message': 'refactor: rename methods and variables\n'}


In [ ]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [ ]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,daducci_amico,00534996cc67dc37a0406ba20f8d16a68fa3ff79,nightwnvol <notte_94@hotmail.it>,1656677387,refactor: rename methods and variables\n


In [ ]:
#mode a means append, so you have all your projects in the same file
yournetid='eabbott9'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github